---

📌 **Set `REPOSITORY_URL` below to your own repository containing these fixes, or upload the fixed project to `/content/Whisper-WebUI`.**

🖋 **Author**: [jhj0517](https://github.com/jhj0517/Whisper-WebUI/blob/master/notebook/whisper-webui.ipynb)

😎 **Support the Project**:

If you find this project useful, please consider supporting it:

❤️ https://github.com/sponsors/jhj0517

---

In [ ]:
#@title #(Optional) Check GPU
#@markdown Some models may not function correctly on a CPU runtime.

#@markdown so you should check your GPU setup before run.
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
print(result.stdout or result.stderr)

In [ ]:
#@title #Installation
#@markdown Enter the URL and branch of YOUR repository after pushing the fixes.
#@markdown Alternatively, upload and extract the fixed project to `/content/Whisper-WebUI` and leave the URL blank.
#@markdown Dependencies are installed into a separate Python environment; no kernel restart is required.
REPOSITORY_URL = 'https://github.com/joelhoty/Whisper_WebUI.git' #@param {type: "string"}
REPOSITORY_REF = '' #@param {type: "string"}
# Leave REPOSITORY_REF blank to use the repository's default branch.

from pathlib import Path
import os
import subprocess
import sys

def run(command, **kwargs):
    # Colab hides output that child processes write directly; relay it so errors and URLs are visible.
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, 'PYTHONUNBUFFERED': '1'}, **kwargs)
    try:
        for line in process.stdout:
            print(line, end='')
    except BaseException:
        process.kill()
        raise
    finally:
        process.stdout.close()
        process.wait()
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

PROJECT_DIR = Path('/content/Whisper-WebUI')
if not PROJECT_DIR.exists():
    if not REPOSITORY_URL.strip():
        raise ValueError('Set REPOSITORY_URL to your fixed repository, or upload the fixed project first.')
    clone_command = ['git', 'clone']
    if REPOSITORY_REF.strip():
        clone_command.extend(['--branch', REPOSITORY_REF.strip()])
    run([*clone_command, '--', REPOSITORY_URL.strip(), str(PROJECT_DIR)])
elif (PROJECT_DIR / '.git').exists():
    origin = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if REPOSITORY_URL and origin.rstrip('/').removesuffix('.git') != REPOSITORY_URL.rstrip('/').removesuffix('.git'):
        raise ValueError(f'Existing checkout belongs to {origin}. Start a fresh runtime to use another repository.')
    print(f'Reusing existing checkout: {origin}')
    try:
        # Pick up fixes pushed after this runtime first cloned the repository.
        run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'])
    except subprocess.CalledProcessError:
        print('Could not update the existing checkout; continuing with it as is.')
    run(['git', '-C', str(PROJECT_DIR), 'log', '-1', '--oneline'])

bootstrap = PROJECT_DIR / 'notebook' / 'colab_runtime.py'
if not bootstrap.is_file():
    raise RuntimeError('This checkout does not contain the Colab fixes. Push all changed files to your repository first.')
run([sys.executable, str(bootstrap), 'install'], cwd=PROJECT_DIR)


In [ ]:
#@title # (Optional) Mount Google Drive
#@markdown Uploading large input files directly via UI may consume alot of time because it has to be uploaded in colab's server.
#@markdown <br>This section is for using the input file paths from Google Drive to reduce such file uploading time.
#@markdown <br>For example, you can first upload the input file to Google Drive and use the directroy path in the "Input Folder Path" input, as shown below.

#@markdown ![image](https://github.com/user-attachments/assets/85330905-e3ec-4502-bc4b-b9d1c5b41aa2)

#@markdown <br>And it will mount the output paths to your Google Drive's as well. This section is optional and can be ignored.


# Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')


# Symlink Output Paths for Whisper-WebUI
import os

OUTPUT_DIRECTORY_PATH = '/content/drive/MyDrive/Whisper-WebUI/outputs'  # @param {type:"string"}
local_output_path = '/content/Whisper-WebUI/outputs'
os.makedirs(OUTPUT_DIRECTORY_PATH, exist_ok=True)

if os.path.islink(local_output_path):
    if os.path.realpath(local_output_path) != os.path.realpath(OUTPUT_DIRECTORY_PATH):
        os.unlink(local_output_path)
elif os.path.exists(local_output_path):
    import tempfile
    backup_dir = tempfile.mkdtemp(prefix='outputs-before-drive-', dir='/content/Whisper-WebUI')
    os.rename(local_output_path, os.path.join(backup_dir, 'outputs'))
    print(f'Existing local outputs preserved in {backup_dir}')

if not os.path.islink(local_output_path):
    os.symlink(OUTPUT_DIRECTORY_PATH, local_output_path)
print(os.listdir(local_output_path))

In [ ]:
#@title # (Optional) Configure arguments
#@markdown This section is used to configure some command line arguments.

#@markdown You can simply ignore this section and the default values will be used.

USERNAME = '' #@param {type: "string"}
PASSWORD = '' #@param {type: "string"}
WHISPER_TYPE = 'faster-whisper' # @param ["whisper", "faster-whisper", "insanely-fast-whisper"]
THEME = '' #@param {type: "string"}

arguments = []
if USERNAME:
  arguments.extend(['--username', USERNAME])
if PASSWORD:
  arguments.extend(['--password', PASSWORD])
if THEME:
  arguments.extend(['--theme', THEME])
if WHISPER_TYPE:
  arguments.extend(['--whisper_type', WHISPER_TYPE])


#@markdown If you wonder how these arguments are used, you can see the [Wiki](https://github.com/jhj0517/Whisper-WebUI/wiki/Command-Line-Arguments).

In [ ]:
#@title #Run
#@markdown Once the installation is complete, you can use public URL that is displayed.
from pathlib import Path
import os
import subprocess
import sys

def run(command, **kwargs):
    # Colab hides output that child processes write directly; relay it so errors and URLs are visible.
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, 'PYTHONUNBUFFERED': '1'}, **kwargs)
    try:
        for line in process.stdout:
            print(line, end='')
    except BaseException:
        process.kill()
        raise
    finally:
        process.stdout.close()
        process.wait()
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

PROJECT_DIR = Path('/content/Whisper-WebUI')
bootstrap = PROJECT_DIR / 'notebook' / 'colab_runtime.py'
if not bootstrap.is_file():
    raise RuntimeError('Run the Installation cell with your fixed repository first.')
run([sys.executable, str(bootstrap), 'run', *globals().get('arguments', [])], cwd=PROJECT_DIR)